# Week 2, §2.3 — Mapping the Pareto-improving region

Two parties $L, R$ have ideal points and salience weights over a
two-dimensional policy space $(x, y) \in [0,1]^2$. We map the set of
bundles that strictly improve on the majority-rule default for both.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(2026)


In [ ]:
def utility(xy, ideal, alpha):
    '''Weighted Euclidean utility with salience alpha on dimension x.'''
    return -(alpha * (xy[..., 0] - ideal[0]) ** 2
             + (1 - alpha) * (xy[..., 1] - ideal[1]) ** 2)

def pareto_region(alpha_L, alpha_R, ideal_L=(0.2, 1.0), ideal_R=(1.0, 0.2),
                  default=(0.5, 0.5), grid_n=200):
    xs = np.linspace(0, 1, grid_n)
    ys = np.linspace(0, 1, grid_n)
    X, Y = np.meshgrid(xs, ys)
    XY = np.stack([X, Y], axis=-1)
    UL = utility(XY, ideal_L, alpha_L)
    UR = utility(XY, ideal_R, alpha_R)
    UL_def = utility(np.array(default), ideal_L, alpha_L)
    UR_def = utility(np.array(default), ideal_R, alpha_R)
    return X, Y, UL, UR, UL_def, UR_def


## (a) Pareto region for the salience setup ($\alpha_L = 0.3, \alpha_R = 0.7$)


In [ ]:
X, Y, UL, UR, UL_def, UR_def = pareto_region(0.3, 0.7)
pareto = (UL > UL_def) & (UR > UR_def)
print(f'U_L^MR = {UL_def:.4f}, U_R^MR = {UR_def:.4f}')
fig, ax = plt.subplots(figsize=(6, 6))
ax.contourf(X, Y, pareto.astype(int), levels=[-0.5, 0.5, 1.5],
            colors=['white', 'C2'], alpha=0.4)
ax.scatter(0.5, 0.5, color='red', s=80, zorder=5, label='default $(0.5, 0.5)$')
ax.scatter(0.76, 0.76, color='C0', s=80, zorder=5, label='logroll-optimal')
ax.scatter(0.2, 1.0, color='C1', s=80, marker='^', label='$L$')
ax.scatter(1.0, 0.2, color='C3', s=80, marker='v', label='$R$')
ax.set(xlim=(0, 1), ylim=(0, 1), xlabel='$x$', ylabel='$y$',
       title='Pareto-improving region (green)')
ax.legend(); ax.set_aspect('equal'); plt.tight_layout(); plt.show()


## (b) Sweep $\alpha_L \in \{0.1, 0.3, 0.5, 0.7, 0.9\}$


In [ ]:
alphas = [0.1, 0.3, 0.5, 0.7, 0.9]
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
areas = []
for ax, aL in zip(axes, alphas):
    aR = 1 - aL
    X, Y, UL, UR, UL_def, UR_def = pareto_region(aL, aR)
    pareto = (UL > UL_def) & (UR > UR_def)
    ax.contourf(X, Y, pareto.astype(int), levels=[-0.5, 0.5, 1.5],
                colors=['white', 'C2'], alpha=0.5)
    ax.scatter(0.5, 0.5, color='red', s=40)
    ax.set(xlim=(0, 1), ylim=(0, 1), title=f'$\\alpha_L = {aL}$',
           xlabel='$x$', ylabel='$y$')
    ax.set_aspect('equal')
    areas.append((aL, pareto.mean()))
plt.tight_layout(); plt.show()
print('alpha_L | Pareto region area (fraction of unit square)')
for a, area in areas:
    print(f'  {a:.1f}    | {area:.4f}')


## (c) Tracing the L--R Pareto frontier (one parameter setting)


In [ ]:
X, Y, UL, UR, UL_def, UR_def = pareto_region(0.3, 0.7)
# extract Pareto frontier: points whose (U_L, U_R) is non-dominated
pts = np.stack([UL.ravel(), UR.ravel()], axis=-1)
# sort by UL descending, take running max of UR
order = np.argsort(-pts[:, 0])
pts_sorted = pts[order]
max_so_far = -np.inf
frontier = []
for p in pts_sorted:
    if p[1] > max_so_far:
        frontier.append(p); max_so_far = p[1]
frontier = np.array(frontier)
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(pts[::20, 0], pts[::20, 1], s=2, alpha=0.2, color='gray')
ax.plot(frontier[:, 0], frontier[:, 1], 'C0-', linewidth=2, label='Pareto frontier')
ax.scatter(UL_def, UR_def, color='red', s=80, zorder=5, label='default')
ax.scatter(-0.1344, -0.1344, color='C2', s=80, zorder=5, label='logroll-optimal')
ax.set(xlabel='$U_L$', ylabel='$U_R$',
       title=r'L--R Pareto frontier ($\alpha_L = 0.3, \alpha_R = 0.7$)')
ax.legend(); plt.tight_layout(); plt.show()


## (d) Three-legislator spatial setup ($\alpha = 1/2$)

With three legislators and quadratic loss, $M$'s ideal is at the centre. The
L--R Pareto-improving region (vs.\ majority-rule default $(0.5, 0.5)$) is
non-empty, but the sub-region in which $M$ is also better off is empty: any
joint improvement for $L$ and $R$ moves the bundle off centre and harms $M$.


In [ ]:
def euclidean_utility(xy, ideal):
    return -((xy[..., 0] - ideal[0]) ** 2 + (xy[..., 1] - ideal[1]) ** 2)

n = 200
X, Y = np.meshgrid(np.linspace(0, 1, n), np.linspace(0, 1, n))
XY = np.stack([X, Y], axis=-1)
UL = euclidean_utility(XY, (0.2, 1.0))
UM = euclidean_utility(XY, (0.5, 0.5))
UR = euclidean_utility(XY, (1.0, 0.3))
UL_def = -0.34; UM_def = 0.0; UR_def = -0.29
lr = (UL > UL_def) & (UR > UR_def)
all3 = lr & (UM > UM_def)
print(f'L--R region area: {lr.mean():.4f}')
print(f'L+M+R triple-improvement area: {all3.mean():.4f}')
fig, ax = plt.subplots(figsize=(6, 6))
ax.contourf(X, Y, lr.astype(int), levels=[-0.5, 0.5, 1.5],
            colors=['white', 'C2'], alpha=0.4)
if all3.any():
    ax.contourf(X, Y, all3.astype(int), levels=[-0.5, 0.5, 1.5],
                colors=['white', 'gold'], alpha=0.7)
ax.scatter([0.5, 0.6, 0.2, 0.5, 1.0], [0.5, 0.65, 1.0, 0.5, 0.3],
           c=['red', 'C0', 'C1', 'C4', 'C3'], s=80, zorder=5)
ax.set(xlim=(0, 1), ylim=(0, 1), title='Spatial setup: L--R region (green); triple-improvement (gold) is empty')
ax.set_aspect('equal'); plt.tight_layout(); plt.show()


## (e) Take-away

* Salience asymmetry **amplifies** the gains-from-trade (extreme $\alpha_L$
  produces the largest Pareto regions) but is not necessary — even $\alpha_L
  = 0.5$ admits a small Pareto region thanks to spatial divergence.
* The harmed party in the spatial three-legislator case is the median
  legislator $M$, whose veto would otherwise determine the outcome. Parties
  function as commitment devices precisely because they let the L--R
  coalition ignore $M$'s veto.
